# 00 Generate `data_raw` CSVs for HD vs PD HTA project

This notebook generates the raw input CSV files for the Singapore haemodialysis (HD) versus peritoneal dialysis (PD) health economics portfolio project.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Detect project root whether the notebook is run from project root or notebooks/
cwd = Path.cwd()
if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

DATA_RAW = PROJECT_ROOT / "data_raw"
DATA_RAW.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data raw folder:", DATA_RAW)

Project root: /Users/marissa/Desktop/portfolio/hd-pd-singapore-hta
Data raw folder: /Users/marissa/Desktop/portfolio/hd-pd-singapore-hta/data_raw


In [2]:
# Source registry used across the raw CSVs.
# Keep URLs in the CSVs so every value has an auditable origin.

SOURCES = {
    "SG_MTE_METHODS": {
        "source_title": "Singapore Medical Technology Evaluation Methods and Process Guide",
        "source_url": "https://www.ace-hta.gov.sg/resources/process-methods/",
        "data_origin": "Singapore medical technology evaluation methods source"
    },
    "MOH_PD_POLICY": {
        "source_title": "Ministry of Health Singapore. Key causes of recent increase in number of patients requiring kidney dialysis",
        "source_url": "https://www.moh.gov.sg/newsroom/key-causes-of-recent-increase-in-number-of-patients-requiring-kidney-dialysis/",
        "data_origin": "Singapore policy statement on PD-preferred strategy and PD uptake target"
    },
    "NKF_KEY_STATS": {
        "source_title": "National Kidney Foundation Singapore. Key Statistics",
        "source_url": "https://nkfs.org/about-us/key-statistics/",
        "data_origin": "Singapore kidney failure and dialysis burden statistics"
    },
    "HEALTHHUB_PD_COSTS": {
        "source_title": "HealthHub Singapore. Peritoneal Dialysis",
        "source_url": "https://www.healthhub.sg/health-conditions/what-is-peritoneal-dialysis",
        "data_origin": "Singapore patient-facing PD cost information"
    },
    "DUKE_MYKIDNEY_COSTS": {
        "source_title": "Duke-NUS Lien Centre for Palliative Care. myKIDNEY: Comparing treatment options",
        "source_url": "https://www.duke-nus.edu.sg/lcpc/mykidney/treatment-options/comparing-treatment-options",
        "data_origin": "Singapore patient-facing HD and PD cost ranges"
    },
    "YANG_2016_CEA": {
        "source_title": "Yang F, Lau T, Luo N. Cost-effectiveness of haemodialysis and peritoneal dialysis for patients with end-stage renal disease in Singapore. Nephrology. 2016.",
        "source_url": "https://pubmed.ncbi.nlm.nih.gov/26566750/",
        "data_origin": "Singapore peer-reviewed cost-effectiveness model"
    },
    "YANG_2018_PD_HRQoL": {
        "source_title": "Yang F et al. Health-Related Quality of Life in Patients Treated with Continuous Ambulatory Peritoneal Dialysis and Automated Peritoneal Dialysis in Singapore.",
        "source_url": "https://link.springer.com/article/10.1007/s41669-017-0046-z",
        "data_origin": "Singapore PD health-related quality-of-life evidence"
    },
    "KHOO_2022_OUTCOMES": {
        "source_title": "Khoo CY et al. Death and cardiovascular outcomes in end-stage renal failure patients on different modalities of dialysis. Ann Acad Med Singap. 2022.",
        "source_url": "https://annals.edu.sg/death-and-cardiovascular-outcomes-in-end-stage-renal-failure-patients-on-different-modalities-of-dialysis/",
        "data_origin": "Singapore population-based dialysis modality outcome study"
    },
    "KHAN_2022_ECONOMICS": {
        "source_title": "Khan BA et al. Health economics of kidney replacement therapy in Singapore: Taking stock and looking ahead. Ann Acad Med Singap. 2022.",
        "source_url": "https://annals.edu.sg/health-economics-of-kidney-replacement-therapy-in-singapore-taking-stock-and-looking-ahead/",
        "data_origin": "Singapore health economics discussion of kidney replacement therapy"
    },
    "COOPER_2020_UTILITIES": {
        "source_title": "Cooper JT et al. Health related quality of life utility weights for economic evaluation through different stages of chronic kidney disease: a systematic review. 2020.",
        "source_url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC7507735/",
        "data_origin": "Systematic review of CKD health-state utility evidence"
    },
    "DRUMMOND_2015": {
        "source_title": "Drummond MF et al. Methods for the Economic Evaluation of Health Care Programmes. 4th ed. Oxford University Press; 2015.",
        "source_url": "NA",
        "data_origin": "Health economic evaluation textbook"
    }
}

def src(key):
    return SOURCES[key]

def add_source(row, key):
    s = src(key)
    return {
        **row,
        "source_key": key,
        "source_title": s["source_title"],
        "source_url": s["source_url"],
        "data_origin": s["data_origin"]
    }

def write_csv(df, filename):
    path = DATA_RAW / filename
    df.to_csv(path, index=False)
    print(f"Wrote {filename}: {len(df)} rows")
    return path

In [3]:
# 1. model_scope_raw.csv

model_scope_rows = [
    add_source({
        "section": "project_identity",
        "field": "project_title",
        "value": "Economic Evaluation and Budget Impact Analysis of Increased Peritoneal Dialysis Use Compared with Centre-Based Haemodialysis for Kidney Failure in Singapore",
        "unit": "text",
        "notes": "Report title for portfolio model."
    }, "SG_MTE_METHODS"),
    add_source({
        "section": "decision_problem",
        "field": "decision_problem",
        "value": "Should Singapore increase peritoneal dialysis uptake among medically suitable kidney failure patients compared with a haemodialysis-dominant current practice pathway?",
        "unit": "text",
        "notes": "Frames the project as a policy pathway evaluation, not a claim that PD is suitable for all patients."
    }, "MOH_PD_POLICY"),
    add_source({
        "section": "population",
        "field": "population",
        "value": "Adults with kidney failure in Singapore who are medically suitable for either PD or HD",
        "unit": "text",
        "notes": "The intervention-eligible population should exclude patients clinically unsuitable for PD."
    }, "MOH_PD_POLICY"),
    add_source({
        "section": "intervention",
        "field": "intervention",
        "value": "Increased uptake of peritoneal dialysis among medically suitable new dialysis patients",
        "unit": "text",
        "notes": "Modelled as a modality uptake scenario, not as a new drug or device."
    }, "MOH_PD_POLICY"),
    add_source({
        "section": "comparator",
        "field": "comparator",
        "value": "Current practice / haemodialysis-dominant dialysis pathway",
        "unit": "text",
        "notes": "Comparator will be parameterised using modality mix inputs."
    }, "MOH_PD_POLICY"),
    add_source({
        "section": "methods",
        "field": "perspective",
        "value": "Singapore healthcare system perspective",
        "unit": "text",
        "notes": "Direct healthcare costs only in the base case."
    }, "SG_MTE_METHODS"),
    add_source({
        "section": "methods",
        "field": "model_type",
        "value": "Cohort Markov cost-utility model with five-year budget impact analysis",
        "unit": "text",
        "notes": "Markov modelling is appropriate because dialysis is a chronic pathway with repeated state transitions."
    }, "DRUMMOND_2015"),
    add_source({
        "section": "methods",
        "field": "base_case_time_horizon",
        "value": "5",
        "unit": "years",
        "notes": "Five years is aligned with the planned budget impact horizon and keeps the portfolio model manageable."
    }, "SG_MTE_METHODS"),
    add_source({
        "section": "methods",
        "field": "cycle_length",
        "value": "1",
        "unit": "year",
        "notes": "Annual cycles match annual cost and transition inputs used in a first-pass model."
    }, "DRUMMOND_2015"),
    add_source({
        "section": "model_structure",
        "field": "health_states",
        "value": "Haemodialysis; Peritoneal dialysis; Switched from PD to HD; Death",
        "unit": "text",
        "notes": "Transplant is excluded initially and can be added as a scenario."
    }, "DRUMMOND_2015"),
    add_source({
        "section": "outcomes",
        "field": "primary_economic_outcome",
        "value": "Incremental cost per QALY gained",
        "unit": "text",
        "notes": "If utilities remain weak, also present a cost-consequence interpretation."
    }, "DRUMMOND_2015"),
    add_source({
        "section": "outcomes",
        "field": "secondary_outcomes",
        "value": "Total costs; life-years; QALYs; deaths; modality switches; net budget impact",
        "unit": "text",
        "notes": "Outputs required for the economic evaluation and BIA tables."
    }, "SG_MTE_METHODS"),
    add_source({
        "section": "budget_impact",
        "field": "budget_impact_horizon",
        "value": "5",
        "unit": "years",
        "notes": "Compares current practice with low, base and high PD uptake scenarios."
    }, "SG_MTE_METHODS"),
    add_source({
        "section": "policy_anchor",
        "field": "pd_uptake_anchor",
        "value": "19% current anchor among new dialysis patients; 30% policy target by 2025",
        "unit": "percent",
        "notes": "Detailed yearly scenarios are specified in uptake_scenarios_raw.csv."
    }, "MOH_PD_POLICY"),
    add_source({
        "section": "costing",
        "field": "currency",
        "value": "Singapore dollars",
        "unit": "SGD",
        "notes": "Cost year will be reported according to source year and model assumption."
    }, "SG_MTE_METHODS"),
    add_source({
        "section": "status",
        "field": "report_status",
        "value": "Training portfolio model, not an official funding evaluation",
        "unit": "text",
        "notes": "Prevents overclaiming policy authority."
    }, "SG_MTE_METHODS"),
]

model_scope_df = pd.DataFrame(model_scope_rows)
write_csv(model_scope_df, "model_scope_raw.csv")
model_scope_df.head()

Wrote model_scope_raw.csv: 16 rows


,section,field,value,unit,notes,source_key,source_title,source_url,data_origin
0,project_identity,project_title,Economic Evaluation and Budget Impact Analysis...,text,Report title for portfolio model.,SG_MTE_METHODS,Singapore Medical Technology Evaluation Method...,https://www.ace-hta.gov.sg/resources/process-m...,Singapore medical technology evaluation method...
1,decision_problem,decision_problem,Should Singapore increase peritoneal dialysis ...,text,Frames the project as a policy pathway evaluat...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...
2,population,population,Adults with kidney failure in Singapore who ar...,text,The intervention-eligible population should ex...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...
3,intervention,intervention,Increased uptake of peritoneal dialysis among ...,text,"Modelled as a modality uptake scenario, not as...",MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...
4,comparator,comparator,Current practice / haemodialysis-dominant dial...,text,Comparator will be parameterised using modalit...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...


In [4]:
# 2. population_inputs_raw.csv

population_rows = [
    add_source({
        "parameter": "prevalent_dialysis_patients_singapore",
        "base_value": 9000,
        "low_value": 9000,
        "high_value": np.nan,
        "unit": "patients",
        "qualifier": "more_than",
        "use_in_model": "Contextual burden and plausibility check",
        "notes": "NKF reports more than 9,000 dialysis patients in Singapore."
    }, "NKF_KEY_STATS"),
    add_source({
        "parameter": "new_kidney_failure_patients_per_day",
        "base_value": 6,
        "low_value": np.nan,
        "high_value": np.nan,
        "unit": "patients_per_day",
        "qualifier": "approximately",
        "use_in_model": "Incident population anchor",
        "notes": "NKF reports around six new kidney failure patients daily."
    }, "NKF_KEY_STATS"),
    add_source({
        "parameter": "estimated_new_kidney_failure_patients_per_year",
        "base_value": 6 * 365,
        "low_value": np.nan,
        "high_value": np.nan,
        "unit": "patients_per_year",
        "qualifier": "derived",
        "use_in_model": "Annual incident population before PD-suitability adjustment",
        "notes": "Derived as 6 new patients per day multiplied by 365 days. Replace with registry annual incident count if available."
    }, "NKF_KEY_STATS"),
    add_source({
        "parameter": "diabetes_share_of_new_kidney_failure_cases",
        "base_value": 2/3,
        "low_value": np.nan,
        "high_value": np.nan,
        "unit": "proportion",
        "qualifier": "approximately",
        "use_in_model": "Case-mix context and scenario discussion",
        "notes": "NKF reports that about two in three new kidney failure cases are due to diabetes."
    }, "NKF_KEY_STATS"),
    add_source({
        "parameter": "annual_dialysis_spending_singapore",
        "base_value": 300_000_000,
        "low_value": np.nan,
        "high_value": np.nan,
        "unit": "SGD_per_year",
        "qualifier": "approximately",
        "use_in_model": "Macro-level affordability and context",
        "notes": "NKF reports around S$300 million is spent annually on dialysis treatment."
    }, "NKF_KEY_STATS"),
]

population_df = pd.DataFrame(population_rows)
write_csv(population_df, "population_inputs_raw.csv")
population_df

Wrote population_inputs_raw.csv: 5 rows


,parameter,base_value,low_value,high_value,unit,qualifier,use_in_model,notes,source_key,source_title,source_url,data_origin
0,prevalent_dialysis_patients_singapore,9.000000e+03,9000.0,NaN,patients,more_than,Contextual burden and plausibility check,"NKF reports more than 9,000 dialysis patients ...",NKF_KEY_STATS,National Kidney Foundation Singapore. Key Stat...,https://nkfs.org/about-us/key-statistics/,Singapore kidney failure and dialysis burden s...
1,new_kidney_failure_patients_per_day,6.000000e+00,NaN,NaN,patients_per_day,approximately,Incident population anchor,NKF reports around six new kidney failure pati...,NKF_KEY_STATS,National Kidney Foundation Singapore. Key Stat...,https://nkfs.org/about-us/key-statistics/,Singapore kidney failure and dialysis burden s...
2,estimated_new_kidney_failure_patients_per_year,2.190000e+03,NaN,NaN,patients_per_year,derived,Annual incident population before PD-suitabili...,Derived as 6 new patients per day multiplied b...,NKF_KEY_STATS,National Kidney Foundation Singapore. Key Stat...,https://nkfs.org/about-us/key-statistics/,Singapore kidney failure and dialysis burden s...
3,diabetes_share_of_new_kidney_failure_cases,6.666667e-01,NaN,NaN,proportion,approximately,Case-mix context and scenario discussion,NKF reports that about two in three new kidney...,NKF_KEY_STATS,National Kidney Foundation Singapore. Key Stat...,https://nkfs.org/about-us/key-statistics/,Singapore kidney failure and dialysis burden s...
4,annual_dialysis_spending_singapore,3.000000e+08,NaN,NaN,SGD_per_year,approximately,Macro-level affordability and context,NKF reports around S$300 million is spent annu...,NKF_KEY_STATS,National Kidney Foundation Singapore. Key Stat...,https://nkfs.org/about-us/key-statistics/,Singapore kidney failure and dialysis burden s...


In [5]:
# 3. modality_mix_raw.csv

modality_rows = [
    add_source({
        "parameter": "current_pd_uptake_new_dialysis_patients",
        "base_value": 0.19,
        "low_value": np.nan,
        "high_value": np.nan,
        "unit": "proportion",
        "qualifier": "policy_anchor",
        "use_in_model": "Current practice uptake anchor",
        "notes": "MOH reported PD uptake among new dialysis patients at 19% before the planned increase."
    }, "MOH_PD_POLICY"),
    add_source({
        "parameter": "target_pd_uptake_new_dialysis_patients",
        "base_value": 0.30,
        "low_value": np.nan,
        "high_value": np.nan,
        "unit": "proportion",
        "qualifier": "policy_target",
        "use_in_model": "Base-case adoption target",
        "notes": "MOH stated a target to raise PD uptake among new dialysis patients to 30% by 2025."
    }, "MOH_PD_POLICY"),
    add_source({
        "parameter": "current_hd_uptake_new_dialysis_patients",
        "base_value": 0.81,
        "low_value": np.nan,
        "high_value": np.nan,
        "unit": "proportion",
        "qualifier": "derived",
        "use_in_model": "Current practice comparator mix",
        "notes": "Derived as 1 minus current PD uptake."
    }, "MOH_PD_POLICY"),
    add_source({
        "parameter": "target_hd_uptake_new_dialysis_patients",
        "base_value": 0.70,
        "low_value": np.nan,
        "high_value": np.nan,
        "unit": "proportion",
        "qualifier": "derived",
        "use_in_model": "New scenario comparator mix",
        "notes": "Derived as 1 minus target PD uptake."
    }, "MOH_PD_POLICY"),
    add_source({
        "parameter": "apd_share_of_pd_patients",
        "base_value": 0.74,
        "low_value": 0.60,
        "high_value": 0.85,
        "unit": "proportion",
        "qualifier": "literature_based",
        "use_in_model": "Blended PD cost calculation",
        "notes": "Used to estimate blended PD cost where PD is not separated into APD and CAPD in the main model."
    }, "KHAN_2022_ECONOMICS"),
    add_source({
        "parameter": "capd_share_of_pd_patients",
        "base_value": 0.26,
        "low_value": 0.15,
        "high_value": 0.40,
        "unit": "proportion",
        "qualifier": "literature_based",
        "use_in_model": "Blended PD cost calculation",
        "notes": "Complements APD share. Values should sum to 1 in the base case."
    }, "KHAN_2022_ECONOMICS"),
]

modality_df = pd.DataFrame(modality_rows)
write_csv(modality_df, "modality_mix_raw.csv")
modality_df

Wrote modality_mix_raw.csv: 6 rows


,parameter,base_value,low_value,high_value,unit,qualifier,use_in_model,notes,source_key,source_title,source_url,data_origin
0,current_pd_uptake_new_dialysis_patients,0.19,NaN,NaN,proportion,policy_anchor,Current practice uptake anchor,MOH reported PD uptake among new dialysis pati...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...
1,target_pd_uptake_new_dialysis_patients,0.30,NaN,NaN,proportion,policy_target,Base-case adoption target,MOH stated a target to raise PD uptake among n...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...
2,current_hd_uptake_new_dialysis_patients,0.81,NaN,NaN,proportion,derived,Current practice comparator mix,Derived as 1 minus current PD uptake.,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...
3,target_hd_uptake_new_dialysis_patients,0.70,NaN,NaN,proportion,derived,New scenario comparator mix,Derived as 1 minus target PD uptake.,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...
4,apd_share_of_pd_patients,0.74,0.60,0.85,proportion,literature_based,Blended PD cost calculation,Used to estimate blended PD cost where PD is n...,KHAN_2022_ECONOMICS,Khan BA et al. Health economics of kidney repl...,https://annals.edu.sg/health-economics-of-kidn...,Singapore health economics discussion of kidne...
5,capd_share_of_pd_patients,0.26,0.15,0.40,proportion,literature_based,Blended PD cost calculation,Complements APD share. Values should sum to 1 ...,KHAN_2022_ECONOMICS,Khan BA et al. Health economics of kidney repl...,https://annals.edu.sg/health-economics-of-kidn...,Singapore health economics discussion of kidne...


In [6]:
# 4. clinical_outcomes_evidence_raw.csv

clinical_rows = [
    add_source({
        "study_or_source": "MOH policy statement",
        "evidence_type": "policy_summary",
        "population": "Medically suitable kidney failure patients requiring dialysis in Singapore",
        "comparison": "PD versus HD",
        "outcome": "Clinical comparability and cost-effectiveness statement",
        "hd_value": "NA",
        "pd_value": "NA",
        "effect_estimate": "MOH states PD has comparable clinical outcomes and is more cost-effective for medically suitable patients",
        "use_in_model": "Justifies scenario question, not used as a transition probability",
        "notes": "This is policy framing, not a trial estimate."
    }, "MOH_PD_POLICY"),
    add_source({
        "study_or_source": "Khoo et al. 2022",
        "evidence_type": "retrospective population-based cohort",
        "population": "Adult patients initiated on dialysis in Singapore between 2007 and 2012",
        "comparison": "Initial HD versus initial PD",
        "outcome": "Cohort size by modality",
        "hd_value": 4449,
        "pd_value": 860,
        "effect_estimate": "Total cohort n=5,309",
        "use_in_model": "Clinical evidence context and modality outcome discussion",
        "notes": "The cohort is observational, so modality differences may reflect selection and case-mix."
    }, "KHOO_2022_OUTCOMES"),
    add_source({
        "study_or_source": "Khoo et al. 2022",
        "evidence_type": "retrospective population-based cohort",
        "population": "Adult patients initiated on dialysis in Singapore between 2007 and 2012",
        "comparison": "Initial HD versus initial PD",
        "outcome": "All-cause death during follow-up",
        "hd_value": "NA",
        "pd_value": "NA",
        "effect_estimate": "Overall all-cause death incidence 34%",
        "use_in_model": "Mortality calibration context",
        "notes": "Do not directly convert to annual death probability without follow-up distribution."
    }, "KHOO_2022_OUTCOMES"),
    add_source({
        "study_or_source": "Khoo et al. 2022",
        "evidence_type": "retrospective population-based cohort",
        "population": "Adult patients initiated on dialysis in Singapore between 2007 and 2012",
        "comparison": "Initial PD versus initial HD",
        "outcome": "All-cause mortality hazard ratio",
        "hd_value": "reference",
        "pd_value": "HR 1.51",
        "effect_estimate": "HR 1.51, 95% CI 1.35 to 1.70",
        "use_in_model": "Scenario analysis only",
        "notes": "Base case should be cautious because observational estimates may be confounded by patient selection."
    }, "KHOO_2022_OUTCOMES"),
    add_source({
        "study_or_source": "Yang et al. 2016",
        "evidence_type": "cost-effectiveness model",
        "population": "Hypothetical 60-year-old non-diabetic ESRD cohort in Singapore",
        "comparison": "CAPD, APD and HD",
        "outcome": "QALYs in base-case model",
        "hd_value": 4.69,
        "pd_value": "CAPD 3.27; APD 3.48",
        "effect_estimate": "CAPD had the highest probability of being cost-effective at the cited WTP threshold",
        "use_in_model": "Reference model and utility/QALY plausibility check",
        "notes": "Do not copy the model result mechanically because this project has a different policy question and time horizon."
    }, "YANG_2016_CEA"),
]

clinical_df = pd.DataFrame(clinical_rows)
write_csv(clinical_df, "clinical_outcomes_evidence_raw.csv")
clinical_df

Wrote clinical_outcomes_evidence_raw.csv: 5 rows


,study_or_source,evidence_type,population,comparison,outcome,hd_value,pd_value,effect_estimate,use_in_model,notes,source_key,source_title,source_url,data_origin
0,MOH policy statement,policy_summary,Medically suitable kidney failure patients req...,PD versus HD,Clinical comparability and cost-effectiveness ...,NA,NA,MOH states PD has comparable clinical outcomes...,"Justifies scenario question, not used as a tra...","This is policy framing, not a trial estimate.",MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...
1,Khoo et al. 2022,retrospective population-based cohort,Adult patients initiated on dialysis in Singap...,Initial HD versus initial PD,Cohort size by modality,4449,860,"Total cohort n=5,309",Clinical evidence context and modality outcome...,"The cohort is observational, so modality diffe...",KHOO_2022_OUTCOMES,Khoo CY et al. Death and cardiovascular outcom...,https://annals.edu.sg/death-and-cardiovascular...,Singapore population-based dialysis modality o...
2,Khoo et al. 2022,retrospective population-based cohort,Adult patients initiated on dialysis in Singap...,Initial HD versus initial PD,All-cause death during follow-up,NA,NA,Overall all-cause death incidence 34%,Mortality calibration context,Do not directly convert to annual death probab...,KHOO_2022_OUTCOMES,Khoo CY et al. Death and cardiovascular outcom...,https://annals.edu.sg/death-and-cardiovascular...,Singapore population-based dialysis modality o...
3,Khoo et al. 2022,retrospective population-based cohort,Adult patients initiated on dialysis in Singap...,Initial PD versus initial HD,All-cause mortality hazard ratio,reference,HR 1.51,"HR 1.51, 95% CI 1.35 to 1.70",Scenario analysis only,Base case should be cautious because observati...,KHOO_2022_OUTCOMES,Khoo CY et al. Death and cardiovascular outcom...,https://annals.edu.sg/death-and-cardiovascular...,Singapore population-based dialysis modality o...
4,Yang et al. 2016,cost-effectiveness model,Hypothetical 60-year-old non-diabetic ESRD coh...,"CAPD, APD and HD",QALYs in base-case model,4.69,CAPD 3.27; APD 3.48,CAPD had the highest probability of being cost...,Reference model and utility/QALY plausibility ...,Do not copy the model result mechanically beca...,YANG_2016_CEA,"Yang F, Lau T, Luo N. Cost-effectiveness of ha...",https://pubmed.ncbi.nlm.nih.gov/26566750/,Singapore peer-reviewed cost-effectiveness model


In [7]:
# 5. cost_inputs_raw.csv

# Base values use midpoints where sources provide ranges.
hd_monthly_base = (2800 + 3500) / 2
capd_monthly_base = (1100 + 1300) / 2
apd_monthly_base = (1600 + 1800) / 2

apd_share = 0.74
capd_share = 0.26
pd_blended_monthly_base = apd_share * apd_monthly_base + capd_share * capd_monthly_base

cost_rows = [
    add_source({
        "parameter": "hd_monthly_cost",
        "modality": "HD",
        "base_value": hd_monthly_base,
        "low_value": 2800,
        "high_value": 3500,
        "unit": "SGD_per_month",
        "qualifier": "midpoint_of_range",
        "include_in_base_case": True,
        "notes": "Patient-facing monthly HD cost range, excluding transport costs. This is a cost proxy, not a full provider cost."
    }, "DUKE_MYKIDNEY_COSTS"),
    add_source({
        "parameter": "hd_annual_cost",
        "modality": "HD",
        "base_value": hd_monthly_base * 12,
        "low_value": 2800 * 12,
        "high_value": 3500 * 12,
        "unit": "SGD_per_year",
        "qualifier": "derived_from_monthly",
        "include_in_base_case": True,
        "notes": "Derived from monthly HD cost."
    }, "DUKE_MYKIDNEY_COSTS"),
    add_source({
        "parameter": "capd_monthly_cost",
        "modality": "CAPD",
        "base_value": capd_monthly_base,
        "low_value": 1100,
        "high_value": 1300,
        "unit": "SGD_per_month",
        "qualifier": "midpoint_of_range",
        "include_in_base_case": False,
        "notes": "Patient-facing CAPD cost range from HealthHub."
    }, "HEALTHHUB_PD_COSTS"),
    add_source({
        "parameter": "capd_annual_cost",
        "modality": "CAPD",
        "base_value": capd_monthly_base * 12,
        "low_value": 1100 * 12,
        "high_value": 1300 * 12,
        "unit": "SGD_per_year",
        "qualifier": "derived_from_monthly",
        "include_in_base_case": False,
        "notes": "Derived from monthly CAPD cost."
    }, "HEALTHHUB_PD_COSTS"),
    add_source({
        "parameter": "apd_monthly_cost",
        "modality": "APD",
        "base_value": apd_monthly_base,
        "low_value": 1600,
        "high_value": 1800,
        "unit": "SGD_per_month",
        "qualifier": "midpoint_of_range",
        "include_in_base_case": False,
        "notes": "Patient-facing APD cost range, excluding electricity bill."
    }, "HEALTHHUB_PD_COSTS"),
    add_source({
        "parameter": "apd_annual_cost",
        "modality": "APD",
        "base_value": apd_monthly_base * 12,
        "low_value": 1600 * 12,
        "high_value": 1800 * 12,
        "unit": "SGD_per_year",
        "qualifier": "derived_from_monthly",
        "include_in_base_case": False,
        "notes": "Derived from monthly APD cost."
    }, "HEALTHHUB_PD_COSTS"),
    add_source({
        "parameter": "pd_blended_monthly_cost",
        "modality": "PD_blended",
        "base_value": pd_blended_monthly_base,
        "low_value": 0.60 * 1600 + 0.40 * 1100,
        "high_value": 0.85 * 1800 + 0.15 * 1300,
        "unit": "SGD_per_month",
        "qualifier": "derived_weighted_average",
        "include_in_base_case": True,
        "notes": "Weighted average using APD/CAPD mix from Singapore health economics article and PD cost ranges from HealthHub."
    }, "KHAN_2022_ECONOMICS"),
    add_source({
        "parameter": "pd_blended_annual_cost",
        "modality": "PD_blended",
        "base_value": pd_blended_monthly_base * 12,
        "low_value": (0.60 * 1600 + 0.40 * 1100) * 12,
        "high_value": (0.85 * 1800 + 0.15 * 1300) * 12,
        "unit": "SGD_per_year",
        "qualifier": "derived_weighted_average",
        "include_in_base_case": True,
        "notes": "Base annual PD cost used in the simplified two-modality model."
    }, "KHAN_2022_ECONOMICS"),
    add_source({
        "parameter": "yang_2016_total_cost_capd",
        "modality": "CAPD",
        "base_value": 169872,
        "low_value": np.nan,
        "high_value": np.nan,
        "unit": "SGD_total_model_horizon",
        "qualifier": "published_model_result",
        "include_in_base_case": False,
        "notes": "Used as external plausibility check, not as the direct annual cost input."
    }, "YANG_2016_CEA"),
    add_source({
        "parameter": "yang_2016_total_cost_apd",
        "modality": "APD",
        "base_value": 201509,
        "low_value": np.nan,
        "high_value": np.nan,
        "unit": "SGD_total_model_horizon",
        "qualifier": "published_model_result",
        "include_in_base_case": False,
        "notes": "Used as external plausibility check, not as the direct annual cost input."
    }, "YANG_2016_CEA"),
    add_source({
        "parameter": "yang_2016_total_cost_hd",
        "modality": "HD",
        "base_value": 306827,
        "low_value": np.nan,
        "high_value": np.nan,
        "unit": "SGD_total_model_horizon",
        "qualifier": "published_model_result",
        "include_in_base_case": False,
        "notes": "Used as external plausibility check, not as the direct annual cost input."
    }, "YANG_2016_CEA"),
]

cost_df = pd.DataFrame(cost_rows)
write_csv(cost_df, "cost_inputs_raw.csv")
cost_df

Wrote cost_inputs_raw.csv: 11 rows


,parameter,modality,base_value,low_value,high_value,unit,qualifier,include_in_base_case,notes,source_key,source_title,source_url,data_origin
0,hd_monthly_cost,HD,3150.0,2800.0,3500.0,SGD_per_month,midpoint_of_range,True,"Patient-facing monthly HD cost range, excludin...",DUKE_MYKIDNEY_COSTS,Duke-NUS Lien Centre for Palliative Care. myKI...,https://www.duke-nus.edu.sg/lcpc/mykidney/trea...,Singapore patient-facing HD and PD cost ranges
1,hd_annual_cost,HD,37800.0,33600.0,42000.0,SGD_per_year,derived_from_monthly,True,Derived from monthly HD cost.,DUKE_MYKIDNEY_COSTS,Duke-NUS Lien Centre for Palliative Care. myKI...,https://www.duke-nus.edu.sg/lcpc/mykidney/trea...,Singapore patient-facing HD and PD cost ranges
2,capd_monthly_cost,CAPD,1200.0,1100.0,1300.0,SGD_per_month,midpoint_of_range,False,Patient-facing CAPD cost range from HealthHub.,HEALTHHUB_PD_COSTS,HealthHub Singapore. Peritoneal Dialysis,https://www.healthhub.sg/health-conditions/wha...,Singapore patient-facing PD cost information
3,capd_annual_cost,CAPD,14400.0,13200.0,15600.0,SGD_per_year,derived_from_monthly,False,Derived from monthly CAPD cost.,HEALTHHUB_PD_COSTS,HealthHub Singapore. Peritoneal Dialysis,https://www.healthhub.sg/health-conditions/wha...,Singapore patient-facing PD cost information
4,apd_monthly_cost,APD,1700.0,1600.0,1800.0,SGD_per_month,midpoint_of_range,False,"Patient-facing APD cost range, excluding elect...",HEALTHHUB_PD_COSTS,HealthHub Singapore. Peritoneal Dialysis,https://www.healthhub.sg/health-conditions/wha...,Singapore patient-facing PD cost information
5,apd_annual_cost,APD,20400.0,19200.0,21600.0,SGD_per_year,derived_from_monthly,False,Derived from monthly APD cost.,HEALTHHUB_PD_COSTS,HealthHub Singapore. Peritoneal Dialysis,https://www.healthhub.sg/health-conditions/wha...,Singapore patient-facing PD cost information
6,pd_blended_monthly_cost,PD_blended,1570.0,1400.0,1725.0,SGD_per_month,derived_weighted_average,True,Weighted average using APD/CAPD mix from Singa...,KHAN_2022_ECONOMICS,Khan BA et al. Health economics of kidney repl...,https://annals.edu.sg/health-economics-of-kidn...,Singapore health economics discussion of kidne...
7,pd_blended_annual_cost,PD_blended,18840.0,16800.0,20700.0,SGD_per_year,derived_weighted_average,True,Base annual PD cost used in the simplified two...,KHAN_2022_ECONOMICS,Khan BA et al. Health economics of kidney repl...,https://annals.edu.sg/health-economics-of-kidn...,Singapore health economics discussion of kidne...
8,yang_2016_total_cost_capd,CAPD,169872.0,NaN,NaN,SGD_total_model_horizon,published_model_result,False,"Used as external plausibility check, not as th...",YANG_2016_CEA,"Yang F, Lau T, Luo N. Cost-effectiveness of ha...",https://pubmed.ncbi.nlm.nih.gov/26566750/,Singapore peer-reviewed cost-effectiveness model
9,yang_2016_total_cost_apd,APD,201509.0,NaN,NaN,SGD_total_model_horizon,published_model_result,False,"Used as external plausibility check, not as th...",YANG_2016_CEA,"Yang F, Lau T, Luo N. Cost-effectiveness of ha...",https://pubmed.ncbi.nlm.nih.gov/26566750/,Singapore peer-reviewed cost-effectiveness model


In [8]:
# 6. utility_inputs_raw.csv

utility_rows = [
    add_source({
        "parameter": "yang_2016_qaly_capd",
        "modality": "CAPD",
        "base_value": 3.27,
        "low_value": np.nan,
        "high_value": np.nan,
        "unit": "QALYs_total_model_horizon",
        "qualifier": "published_model_result",
        "include_in_base_case": False,
        "notes": "Used for plausibility checking against this project's model outputs."
    }, "YANG_2016_CEA"),
    add_source({
        "parameter": "yang_2016_qaly_apd",
        "modality": "APD",
        "base_value": 3.48,
        "low_value": np.nan,
        "high_value": np.nan,
        "unit": "QALYs_total_model_horizon",
        "qualifier": "published_model_result",
        "include_in_base_case": False,
        "notes": "Used for plausibility checking against this project's model outputs."
    }, "YANG_2016_CEA"),
    add_source({
        "parameter": "yang_2016_qaly_hd",
        "modality": "HD",
        "base_value": 4.69,
        "low_value": np.nan,
        "high_value": np.nan,
        "unit": "QALYs_total_model_horizon",
        "qualifier": "published_model_result",
        "include_in_base_case": False,
        "notes": "Used for plausibility checking against this project's model outputs."
    }, "YANG_2016_CEA"),
    add_source({
        "parameter": "health_state_utility_hd",
        "modality": "HD",
        "base_value": 0.58,
        "low_value": 0.44,
        "high_value": 0.71,
        "unit": "utility_weight",
        "qualifier": "placeholder_literature_range",
        "include_in_base_case": True,
        "notes": "Placeholder utility for first-pass model. Cooper et al. systematic review supports using CKD utility evidence, but this exact base value should be sensitivity-tested."
    }, "COOPER_2020_UTILITIES"),
    add_source({
        "parameter": "health_state_utility_pd",
        "modality": "PD",
        "base_value": 0.62,
        "low_value": 0.53,
        "high_value": 0.72,
        "unit": "utility_weight",
        "qualifier": "placeholder_literature_range",
        "include_in_base_case": True,
        "notes": "Placeholder utility for first-pass model. Keep utility difference small because Singapore PD modality HRQoL evidence suggests differences can be modest."
    }, "YANG_2018_PD_HRQoL"),
    add_source({
        "parameter": "health_state_utility_switched_pd_to_hd",
        "modality": "Switched_PD_to_HD",
        "base_value": 0.56,
        "low_value": 0.44,
        "high_value": 0.71,
        "unit": "utility_weight",
        "qualifier": "analyst_assumption",
        "include_in_base_case": True,
        "notes": "Assumed slightly lower than stable HD due to prior technique failure/switch burden. High-priority sensitivity parameter."
    }, "COOPER_2020_UTILITIES"),
    add_source({
        "parameter": "health_state_utility_death",
        "modality": "Death",
        "base_value": 0.0,
        "low_value": 0.0,
        "high_value": 0.0,
        "unit": "utility_weight",
        "qualifier": "standard_economic_model_convention",
        "include_in_base_case": True,
        "notes": "Death has zero utility in QALY modelling."
    }, "DRUMMOND_2015"),
]

utility_df = pd.DataFrame(utility_rows)
write_csv(utility_df, "utility_inputs_raw.csv")
utility_df

Wrote utility_inputs_raw.csv: 7 rows


,parameter,modality,base_value,low_value,high_value,unit,qualifier,include_in_base_case,notes,source_key,source_title,source_url,data_origin
0,yang_2016_qaly_capd,CAPD,3.27,NaN,NaN,QALYs_total_model_horizon,published_model_result,False,Used for plausibility checking against this pr...,YANG_2016_CEA,"Yang F, Lau T, Luo N. Cost-effectiveness of ha...",https://pubmed.ncbi.nlm.nih.gov/26566750/,Singapore peer-reviewed cost-effectiveness model
1,yang_2016_qaly_apd,APD,3.48,NaN,NaN,QALYs_total_model_horizon,published_model_result,False,Used for plausibility checking against this pr...,YANG_2016_CEA,"Yang F, Lau T, Luo N. Cost-effectiveness of ha...",https://pubmed.ncbi.nlm.nih.gov/26566750/,Singapore peer-reviewed cost-effectiveness model
2,yang_2016_qaly_hd,HD,4.69,NaN,NaN,QALYs_total_model_horizon,published_model_result,False,Used for plausibility checking against this pr...,YANG_2016_CEA,"Yang F, Lau T, Luo N. Cost-effectiveness of ha...",https://pubmed.ncbi.nlm.nih.gov/26566750/,Singapore peer-reviewed cost-effectiveness model
3,health_state_utility_hd,HD,0.58,0.44,0.71,utility_weight,placeholder_literature_range,True,Placeholder utility for first-pass model. Coop...,COOPER_2020_UTILITIES,Cooper JT et al. Health related quality of lif...,https://pmc.ncbi.nlm.nih.gov/articles/PMC7507735/,Systematic review of CKD health-state utility ...
4,health_state_utility_pd,PD,0.62,0.53,0.72,utility_weight,placeholder_literature_range,True,Placeholder utility for first-pass model. Keep...,YANG_2018_PD_HRQoL,Yang F et al. Health-Related Quality of Life i...,https://link.springer.com/article/10.1007/s416...,Singapore PD health-related quality-of-life ev...
5,health_state_utility_switched_pd_to_hd,Switched_PD_to_HD,0.56,0.44,0.71,utility_weight,analyst_assumption,True,Assumed slightly lower than stable HD due to p...,COOPER_2020_UTILITIES,Cooper JT et al. Health related quality of lif...,https://pmc.ncbi.nlm.nih.gov/articles/PMC7507735/,Systematic review of CKD health-state utility ...
6,health_state_utility_death,Death,0.00,0.00,0.00,utility_weight,standard_economic_model_convention,True,Death has zero utility in QALY modelling.,DRUMMOND_2015,Drummond MF et al. Methods for the Economic Ev...,NA,Health economic evaluation textbook


In [9]:
# 7. transition_inputs_raw.csv

transition_rows = [
    add_source({
        "parameter": "annual_death_probability_hd",
        "from_state": "HD",
        "to_state": "Death",
        "base_value": 0.12,
        "low_value": 0.08,
        "high_value": 0.18,
        "unit": "annual_probability",
        "qualifier": "analyst_assumption_for_first_pass_model",
        "include_in_base_case": True,
        "notes": "Base case assumes equal annual mortality for HD and PD to avoid over-interpreting observational modality differences. Replace with registry-calibrated survival if available."
    }, "KHOO_2022_OUTCOMES"),
    add_source({
        "parameter": "annual_death_probability_pd",
        "from_state": "PD",
        "to_state": "Death",
        "base_value": 0.12,
        "low_value": 0.08,
        "high_value": 0.20,
        "unit": "annual_probability",
        "qualifier": "analyst_assumption_for_first_pass_model",
        "include_in_base_case": True,
        "notes": "Base case assumes equal mortality. Scenario analysis can apply the Singapore observational HR for PD versus HD."
    }, "KHOO_2022_OUTCOMES"),
    add_source({
        "parameter": "pd_mortality_hazard_ratio_vs_hd",
        "from_state": "PD",
        "to_state": "Death",
        "base_value": 1.51,
        "low_value": 1.35,
        "high_value": 1.70,
        "unit": "hazard_ratio",
        "qualifier": "observational_evidence",
        "include_in_base_case": False,
        "notes": "Use as scenario only because modality selection and case-mix may confound observational mortality differences."
    }, "KHOO_2022_OUTCOMES"),
    add_source({
        "parameter": "annual_pd_to_hd_switch_probability",
        "from_state": "PD",
        "to_state": "Switched_PD_to_HD",
        "base_value": 0.15,
        "low_value": 0.08,
        "high_value": 0.25,
        "unit": "annual_probability",
        "qualifier": "analyst_assumption",
        "include_in_base_case": True,
        "notes": "Represents PD technique failure or modality switch. Needs validation from registry or local clinical input."
    }, "KHAN_2022_ECONOMICS"),
    add_source({
        "parameter": "annual_hd_to_pd_switch_probability",
        "from_state": "HD",
        "to_state": "PD",
        "base_value": 0.01,
        "low_value": 0.00,
        "high_value": 0.03,
        "unit": "annual_probability",
        "qualifier": "analyst_assumption",
        "include_in_base_case": True,
        "notes": "Small switch probability because the policy question focuses on initial modality uptake among new patients."
    }, "MOH_PD_POLICY"),
    add_source({
        "parameter": "annual_transplant_probability",
        "from_state": "Any_dialysis_state",
        "to_state": "Transplant",
        "base_value": 0.00,
        "low_value": 0.00,
        "high_value": 0.03,
        "unit": "annual_probability",
        "qualifier": "excluded_from_base_case",
        "include_in_base_case": False,
        "notes": "Transplant is excluded from the base model to keep the first version transparent. Add as a scenario later if needed."
    }, "KHAN_2022_ECONOMICS"),
]

transition_df = pd.DataFrame(transition_rows)
write_csv(transition_df, "transition_inputs_raw.csv")
transition_df

Wrote transition_inputs_raw.csv: 6 rows


,parameter,from_state,to_state,base_value,low_value,high_value,unit,qualifier,include_in_base_case,notes,source_key,source_title,source_url,data_origin
0,annual_death_probability_hd,HD,Death,0.12,0.08,0.18,annual_probability,analyst_assumption_for_first_pass_model,True,Base case assumes equal annual mortality for H...,KHOO_2022_OUTCOMES,Khoo CY et al. Death and cardiovascular outcom...,https://annals.edu.sg/death-and-cardiovascular...,Singapore population-based dialysis modality o...
1,annual_death_probability_pd,PD,Death,0.12,0.08,0.20,annual_probability,analyst_assumption_for_first_pass_model,True,Base case assumes equal mortality. Scenario an...,KHOO_2022_OUTCOMES,Khoo CY et al. Death and cardiovascular outcom...,https://annals.edu.sg/death-and-cardiovascular...,Singapore population-based dialysis modality o...
2,pd_mortality_hazard_ratio_vs_hd,PD,Death,1.51,1.35,1.70,hazard_ratio,observational_evidence,False,Use as scenario only because modality selectio...,KHOO_2022_OUTCOMES,Khoo CY et al. Death and cardiovascular outcom...,https://annals.edu.sg/death-and-cardiovascular...,Singapore population-based dialysis modality o...
3,annual_pd_to_hd_switch_probability,PD,Switched_PD_to_HD,0.15,0.08,0.25,annual_probability,analyst_assumption,True,Represents PD technique failure or modality sw...,KHAN_2022_ECONOMICS,Khan BA et al. Health economics of kidney repl...,https://annals.edu.sg/health-economics-of-kidn...,Singapore health economics discussion of kidne...
4,annual_hd_to_pd_switch_probability,HD,PD,0.01,0.00,0.03,annual_probability,analyst_assumption,True,Small switch probability because the policy qu...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...
5,annual_transplant_probability,Any_dialysis_state,Transplant,0.00,0.00,0.03,annual_probability,excluded_from_base_case,False,Transplant is excluded from the base model to ...,KHAN_2022_ECONOMICS,Khan BA et al. Health economics of kidney repl...,https://annals.edu.sg/health-economics-of-kidn...,Singapore health economics discussion of kidne...


In [10]:
# 8. uptake_scenarios_raw.csv

# Annual incident patients are anchored to NKF's approximate six new patients daily.
incident_patients_per_year = 6 * 365

# Not all incident kidney failure patients are medically suitable for PD.
# This assumption must be tested because it is a major BIA driver.
pd_suitable_share_base = 0.60
eligible_patients_per_year_base = round(incident_patients_per_year * pd_suitable_share_base)

uptake_rows = []
for i, year in enumerate(range(2026, 2031), start=1):
    current_practice_pd = 0.19

    # Low/base/high are policy adoption scenarios, not observed forecasts.
    low_pd = [0.21, 0.23, 0.25, 0.26, 0.27][i-1]
    base_pd = [0.22, 0.24, 0.26, 0.28, 0.30][i-1]
    high_pd = [0.25, 0.30, 0.34, 0.37, 0.40][i-1]

    uptake_rows.extend([
        add_source({
            "year": year,
            "scenario": "current_practice",
            "incident_patients_per_year": incident_patients_per_year,
            "pd_suitable_share": pd_suitable_share_base,
            "eligible_patients_per_year": eligible_patients_per_year_base,
            "pd_uptake": current_practice_pd,
            "hd_uptake": 1 - current_practice_pd,
            "notes": "Counterfactual scenario keeps PD uptake constant at the current policy anchor."
        }, "MOH_PD_POLICY"),
        add_source({
            "year": year,
            "scenario": "low_pd_adoption",
            "incident_patients_per_year": incident_patients_per_year,
            "pd_suitable_share": pd_suitable_share_base,
            "eligible_patients_per_year": eligible_patients_per_year_base,
            "pd_uptake": low_pd,
            "hd_uptake": 1 - low_pd,
            "notes": "Conservative increase in PD uptake below the 30% policy target by year 5."
        }, "MOH_PD_POLICY"),
        add_source({
            "year": year,
            "scenario": "base_pd_adoption",
            "incident_patients_per_year": incident_patients_per_year,
            "pd_suitable_share": pd_suitable_share_base,
            "eligible_patients_per_year": eligible_patients_per_year_base,
            "pd_uptake": base_pd,
            "hd_uptake": 1 - base_pd,
            "notes": "Base scenario reaches the 30% PD uptake target by year 5."
        }, "MOH_PD_POLICY"),
        add_source({
            "year": year,
            "scenario": "high_pd_adoption",
            "incident_patients_per_year": incident_patients_per_year,
            "pd_suitable_share": pd_suitable_share_base,
            "eligible_patients_per_year": eligible_patients_per_year_base,
            "pd_uptake": high_pd,
            "hd_uptake": 1 - high_pd,
            "notes": "Ambitious scenario exceeds the 30% target, used to test budget impact under stronger uptake."
        }, "MOH_PD_POLICY"),
    ])

uptake_df = pd.DataFrame(uptake_rows)
write_csv(uptake_df, "uptake_scenarios_raw.csv")
uptake_df.head(10)

Wrote uptake_scenarios_raw.csv: 20 rows


,year,scenario,incident_patients_per_year,pd_suitable_share,eligible_patients_per_year,pd_uptake,hd_uptake,notes,source_key,source_title,source_url,data_origin
0,2026,current_practice,2190,0.6,1314,0.19,0.81,Counterfactual scenario keeps PD uptake consta...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...
1,2026,low_pd_adoption,2190,0.6,1314,0.21,0.79,Conservative increase in PD uptake below the 3...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...
2,2026,base_pd_adoption,2190,0.6,1314,0.22,0.78,Base scenario reaches the 30% PD uptake target...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...
3,2026,high_pd_adoption,2190,0.6,1314,0.25,0.75,"Ambitious scenario exceeds the 30% target, use...",MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...
4,2027,current_practice,2190,0.6,1314,0.19,0.81,Counterfactual scenario keeps PD uptake consta...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...
5,2027,low_pd_adoption,2190,0.6,1314,0.23,0.77,Conservative increase in PD uptake below the 3...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...
6,2027,base_pd_adoption,2190,0.6,1314,0.24,0.76,Base scenario reaches the 30% PD uptake target...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...
7,2027,high_pd_adoption,2190,0.6,1314,0.30,0.70,"Ambitious scenario exceeds the 30% target, use...",MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...
8,2028,current_practice,2190,0.6,1314,0.19,0.81,Counterfactual scenario keeps PD uptake consta...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...
9,2028,low_pd_adoption,2190,0.6,1314,0.25,0.75,Conservative increase in PD uptake below the 3...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...


In [11]:
# 9. assumptions_register_raw.csv

assumption_rows = [
    add_source({
        "assumption_id": "A01",
        "assumption": "Only medically suitable patients are eligible for increased PD uptake.",
        "base_value": "PD suitability share = 0.60",
        "low_value": "0.40",
        "high_value": "0.80",
        "uncertainty_type": "structural and parameter uncertainty",
        "validation_needed": "Local nephrology input or registry data on contraindications and social suitability",
        "model_impact": "High impact on budget impact; does not directly change per-patient ICER.",
        "notes": "This prevents the model from pretending all kidney failure patients can receive PD."
    }, "MOH_PD_POLICY"),
    add_source({
        "assumption_id": "A02",
        "assumption": "Base-case Markov model excludes transplant.",
        "base_value": "Excluded",
        "low_value": "NA",
        "high_value": "Scenario can include transplant",
        "uncertainty_type": "structural uncertainty",
        "validation_needed": "Singapore transplant rates and waiting-time data",
        "model_impact": "May affect long-term cost and QALY results if lifetime horizon is added.",
        "notes": "Acceptable for first-pass five-year dialysis pathway model but should be stated clearly."
    }, "KHAN_2022_ECONOMICS"),
    add_source({
        "assumption_id": "A03",
        "assumption": "Base case assumes equal mortality for HD and PD.",
        "base_value": "HD death probability = PD death probability",
        "low_value": "Scenario: lower PD mortality",
        "high_value": "Scenario: apply PD HR 1.51 versus HD",
        "uncertainty_type": "clinical uncertainty and confounding",
        "validation_needed": "Local adjusted survival analysis by patient eligibility and modality choice",
        "model_impact": "High impact on QALYs and ICER.",
        "notes": "This is deliberately cautious because observational modality comparisons may be confounded."
    }, "KHOO_2022_OUTCOMES"),
    add_source({
        "assumption_id": "A04",
        "assumption": "PD cost is represented as a blended APD/CAPD cost.",
        "base_value": "APD 74%; CAPD 26%",
        "low_value": "APD 60%; CAPD 40%",
        "high_value": "APD 85%; CAPD 15%",
        "uncertainty_type": "cost and service-mix uncertainty",
        "validation_needed": "Current Singapore PD modality mix",
        "model_impact": "Moderate to high impact on PD cost and budget impact.",
        "notes": "Blended costing is simpler than modelling APD and CAPD as separate arms."
    }, "KHAN_2022_ECONOMICS"),
    add_source({
        "assumption_id": "A05",
        "assumption": "Published patient-facing dialysis costs are used as cost proxies.",
        "base_value": "HD and PD cost ranges from Singapore public/sector sources",
        "low_value": "Lower source range",
        "high_value": "Higher source range",
        "uncertainty_type": "costing uncertainty",
        "validation_needed": "Provider cost, subsidy, and claims data if available",
        "model_impact": "High impact on cost difference and BIA.",
        "notes": "Do not describe these as exact provider costs."
    }, "DUKE_MYKIDNEY_COSTS"),
    add_source({
        "assumption_id": "A06",
        "assumption": "Health-state utilities are literature-based placeholders.",
        "base_value": "HD 0.58; PD 0.62; switched 0.56",
        "low_value": "Lower published/assumption range",
        "high_value": "Upper published/assumption range",
        "uncertainty_type": "utility uncertainty",
        "validation_needed": "Singapore EQ-5D or SF-6D dialysis utility inputs by modality",
        "model_impact": "High impact on QALY results.",
        "notes": "Report should avoid false precision and include cost-consequence outputs."
    }, "COOPER_2020_UTILITIES"),
    add_source({
        "assumption_id": "A07",
        "assumption": "Five-year horizon is used for both economic and budget impact outputs.",
        "base_value": "5 years",
        "low_value": "1 year",
        "high_value": "Lifetime scenario",
        "uncertainty_type": "time-horizon uncertainty",
        "validation_needed": "Decision-maker preference and data availability",
        "model_impact": "Longer horizons may increase the importance of survival and switching assumptions.",
        "notes": "Five years is pragmatic for a portfolio model, but not a full lifetime HTA."
    }, "SG_MTE_METHODS"),
    add_source({
        "assumption_id": "A08",
        "assumption": "Annual cycle length is used.",
        "base_value": "1 year",
        "low_value": "6 months",
        "high_value": "1 year",
        "uncertainty_type": "model granularity",
        "validation_needed": "Availability of monthly or quarterly transition data",
        "model_impact": "May affect timing of death, switching, costs and QALYs.",
        "notes": "Annual cycles are acceptable for first-pass modelling but less sensitive to within-year transitions."
    }, "DRUMMOND_2015"),
]

assumptions_df = pd.DataFrame(assumption_rows)
write_csv(assumptions_df, "assumptions_register_raw.csv")
assumptions_df

Wrote assumptions_register_raw.csv: 8 rows


,assumption_id,assumption,base_value,low_value,high_value,uncertainty_type,validation_needed,model_impact,notes,source_key,source_title,source_url,data_origin
0,A01,Only medically suitable patients are eligible ...,PD suitability share = 0.60,0.40,0.80,structural and parameter uncertainty,Local nephrology input or registry data on con...,High impact on budget impact; does not directl...,This prevents the model from pretending all ki...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...
1,A02,Base-case Markov model excludes transplant.,Excluded,NA,Scenario can include transplant,structural uncertainty,Singapore transplant rates and waiting-time data,May affect long-term cost and QALY results if ...,Acceptable for first-pass five-year dialysis p...,KHAN_2022_ECONOMICS,Khan BA et al. Health economics of kidney repl...,https://annals.edu.sg/health-economics-of-kidn...,Singapore health economics discussion of kidne...
2,A03,Base case assumes equal mortality for HD and PD.,HD death probability = PD death probability,Scenario: lower PD mortality,Scenario: apply PD HR 1.51 versus HD,clinical uncertainty and confounding,Local adjusted survival analysis by patient el...,High impact on QALYs and ICER.,This is deliberately cautious because observat...,KHOO_2022_OUTCOMES,Khoo CY et al. Death and cardiovascular outcom...,https://annals.edu.sg/death-and-cardiovascular...,Singapore population-based dialysis modality o...
3,A04,PD cost is represented as a blended APD/CAPD c...,APD 74%; CAPD 26%,APD 60%; CAPD 40%,APD 85%; CAPD 15%,cost and service-mix uncertainty,Current Singapore PD modality mix,Moderate to high impact on PD cost and budget ...,Blended costing is simpler than modelling APD ...,KHAN_2022_ECONOMICS,Khan BA et al. Health economics of kidney repl...,https://annals.edu.sg/health-economics-of-kidn...,Singapore health economics discussion of kidne...
4,A05,Published patient-facing dialysis costs are us...,HD and PD cost ranges from Singapore public/se...,Lower source range,Higher source range,costing uncertainty,"Provider cost, subsidy, and claims data if ava...",High impact on cost difference and BIA.,Do not describe these as exact provider costs.,DUKE_MYKIDNEY_COSTS,Duke-NUS Lien Centre for Palliative Care. myKI...,https://www.duke-nus.edu.sg/lcpc/mykidney/trea...,Singapore patient-facing HD and PD cost ranges
5,A06,Health-state utilities are literature-based pl...,HD 0.58; PD 0.62; switched 0.56,Lower published/assumption range,Upper published/assumption range,utility uncertainty,Singapore EQ-5D or SF-6D dialysis utility inpu...,High impact on QALY results.,Report should avoid false precision and includ...,COOPER_2020_UTILITIES,Cooper JT et al. Health related quality of lif...,https://pmc.ncbi.nlm.nih.gov/articles/PMC7507735/,Systematic review of CKD health-state utility ...
6,A07,Five-year horizon is used for both economic an...,5 years,1 year,Lifetime scenario,time-horizon uncertainty,Decision-maker preference and data availability,Longer horizons may increase the importance of...,"Five years is pragmatic for a portfolio model,...",SG_MTE_METHODS,Singapore Medical Technology Evaluation Method...,https://www.ace-hta.gov.sg/resources/process-m...,Singapore medical technology evaluation method...
7,A08,Annual cycle length is used.,1 year,6 months,1 year,model granularity,Availability of monthly or quarterly transitio...,"May affect timing of death, switching, costs a...",Annual cycles are acceptable for first-pass mo...,DRUMMOND_2015,Drummond MF et al. Methods for the Economic Ev...,NA,Health economic evaluation textbook


In [12]:
# 10. Quick QA checks

expected_files = [
    "model_scope_raw.csv",
    "population_inputs_raw.csv",
    "modality_mix_raw.csv",
    "clinical_outcomes_evidence_raw.csv",
    "cost_inputs_raw.csv",
    "utility_inputs_raw.csv",
    "transition_inputs_raw.csv",
    "uptake_scenarios_raw.csv",
    "assumptions_register_raw.csv",
]

print("Generated files:")
for f in expected_files:
    p = DATA_RAW / f
    status = "OK" if p.exists() else "MISSING"
    n_rows = len(pd.read_csv(p)) if p.exists() else 0
    print(f"{status:8} {f:40} rows={n_rows}")

# Check that the base APD/CAPD shares sum to 1.
mix = pd.read_csv(DATA_RAW / "modality_mix_raw.csv")
share_sum = mix.loc[mix["parameter"].isin(["apd_share_of_pd_patients", "capd_share_of_pd_patients"]), "base_value"].astype(float).sum()
print("\nBase APD + CAPD share:", round(share_sum, 4))

# Check current and target uptake values.
uptake = pd.read_csv(DATA_RAW / "uptake_scenarios_raw.csv")
print("\nUptake scenarios:")
print(uptake.groupby("scenario")["pd_uptake"].agg(["min", "max"]).round(3))

# Show created folder content
print("\nCSV files in data_raw:")
for p in sorted(DATA_RAW.glob("*.csv")):
    print("-", p.name)

Generated files:
OK       model_scope_raw.csv                      rows=16
OK       population_inputs_raw.csv                rows=5
OK       modality_mix_raw.csv                     rows=6
OK       clinical_outcomes_evidence_raw.csv       rows=5
OK       cost_inputs_raw.csv                      rows=11
OK       utility_inputs_raw.csv                   rows=7
OK       transition_inputs_raw.csv                rows=6
OK       uptake_scenarios_raw.csv                 rows=20
OK       assumptions_register_raw.csv             rows=8

Base APD + CAPD share: 1.0

Uptake scenarios:
                   min   max
scenario                    
base_pd_adoption  0.22  0.30
current_practice  0.19  0.19
high_pd_adoption  0.25  0.40
low_pd_adoption   0.21  0.27

CSV files in data_raw:
- assumptions_register_raw.csv
- clinical_outcomes_evidence_raw.csv
- cost_inputs_raw.csv
- modality_mix_raw.csv
- model_scope_raw.csv
- population_inputs_raw.csv
- transition_inputs_raw.csv
- uptake_scenarios_raw.csv
- ut